In [12]:
!pip install sentence-transformers stanza vaderSentiment scikit-learn pandas numpy torch transformers
!pip install --upgrade sentence-transformers


In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!ls "/content/drive/MyDrive/Colab_Notebooks/a5"


Mounted at /content/drive
result_v8.csv		       train_split.csv	val_split.csv
result_v8_no_vader_no_nlp.csv  v8_da_model.pt	vBERT_finetune.csv
test_no_answer_2022.csv        v8_ft_model.pt
train_2022.csv		       v8_result


## Imports

In [14]:
import copy
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader, Dataset as TorchDataset
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score, mean_squared_error,
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
)
from sklearn.pipeline import Pipeline


## Config & Constants

In [ ]:
base_path = "/content/drive/MyDrive/Colab_Notebooks/a5/"

CFG = {
    # I/O
    "input":        base_path + "train_2022.csv",
    "output":       base_path + "row_scores_v8_no_vader_no_nlp.csv",
    "test":         base_path + "test_no_answer_2022.csv",
    "test_output":  base_path + "result_v8_no_vader_no_nlp.csv",

    # ML model: "rf" | "elastic" | "gbm"
    "model":        "rf",

    # Phase 1 — MLM Domain Adaptation
    "skip_da":      False,
    "load_da":      None,          # set to a path to skip training and load a saved model
    "da_epochs":    3,
    "da_batch":     8,
    "da_lr":        3e-5,
    "da_mlm_prob":  0.15,
    "da_save":      base_path + "v8_da_model_v2.pt",

    # Phase 1 only mode — set True to exit after MLM adaptation
    "skip_ft":      False,

    # Phase 2 — Supervised Fine-tuning (run inside each CV round)
    "ft_epochs":    3,
    "ft_lr":        2e-5,
    "ft_batch":     16,
    "ft_pairs":     3000,
    "ft_warmup":    100,
    "ft_save":      base_path + "v8_ft_model_v2.pt",
}

SBERT_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
SBERT_DIM        = 768

AGG_FEATURES = [f"emb_{i}" for i in range(SBERT_DIM)]
ALL_FEATURES = AGG_FEATURES


## Step 0 — Load Data

In [16]:
train_df = pd.read_csv(CFG["input"])
train_df["row_id"] = train_df["row_id"].astype(int)
train_df["LABEL"]  = train_df["LABEL"].astype(int)
print(f"Loaded {len(train_df)} training rows")

test_df = pd.read_csv(CFG["test"])
test_df["row_id"] = test_df["row_id"].astype(int)
if "LABEL" not in test_df.columns:
    test_df["LABEL"] = -1
print(f"Loaded {len(test_df)} test rows")


Loaded 2000 training rows
Loaded 11000 test rows


## Step 1 — Build Base SBERT Encoder

In [17]:
def _build_sbert_model() -> SentenceTransformer:
    print(f"  Loading '{SBERT_MODEL_NAME}' …")
    model = SentenceTransformer(SBERT_MODEL_NAME)
    print(f"  Embedding dim: {SBERT_DIM}")
    return model


sbert = _build_sbert_model()


  Loading 'sentence-transformers/all-mpnet-base-v2' …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding dim: 768


## Step 2 — Phase 1: MLM Domain Adaptation

使用 Train + Test 全部文字（無標籤）對 SBERT encoder 進行 MLM 預訓練，
讓模型熟悉本任務的語料分布。

**CFG keys used:** `skip_da`, `load_da`, `da_epochs`, `da_batch`, `da_lr`, `da_mlm_prob`, `da_save`

In [18]:
# ── Helper: lightweight PyTorch dataset for tokenized encodings ──────────
class _TextDataset(TorchDataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


# ── Helper: MLM domain adaptation ────────────────────────────────────────────
def domain_adapt_mlm(
    sbert: SentenceTransformer,
    all_texts: list,
    epochs: int     = 3,
    batch_size: int = 16,
    lr: float       = 3e-5,
    mlm_prob: float = 0.15,
    output_path=None,
) -> None:
    from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling

    print(f"  Texts    : {len(all_texts)} (Train + Test, no labels)")
    print(f"  Epochs   : {epochs}  |  batch={batch_size}  |  lr={lr}")
    print(f"  MLM prob : {mlm_prob}")

    transformer_module = sbert[0]
    hf_model_name      = transformer_module.auto_model.config.name_or_path

    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    mlm_model = AutoModelForMaskedLM.from_pretrained(hf_model_name)
    mlm_model.mpnet.load_state_dict(transformer_module.auto_model.state_dict(), strict=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mlm_model.to(device).train()

    encodings  = tokenizer(all_texts, truncation=True, padding="max_length",
                           max_length=128, return_tensors="pt")
    dataset    = _TextDataset(encodings)
    collator   = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                                  mlm_probability=mlm_prob)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)

    optimizer    = torch.optim.AdamW(mlm_model.parameters(), lr=lr)
    total_steps  = len(dataloader) * epochs
    warmup_steps = max(1, total_steps // 10)
    scheduler    = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps
    )

    global_step = 0
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for step, batch in enumerate(dataloader, 1):
            batch  = {k: v.to(device) for k, v in batch.items()}
            loss   = mlm_model(**batch).loss
            loss.backward()
            optimizer.step()
            if global_step < warmup_steps:
                scheduler.step()
            optimizer.zero_grad()
            total_loss  += loss.item()
            global_step += 1
            if step % 50 == 0:
                print(f"    epoch {epoch}/{epochs}  step {step}/{len(dataloader)}"
                      f"  loss={total_loss/step:.4f}")
        print(f"  Epoch {epoch} done — avg loss: {total_loss/len(dataloader):.4f}")

    mlm_model.to("cpu")
    transformer_module.auto_model.load_state_dict(mlm_model.mpnet.state_dict(), strict=False)
    if output_path:
        sbert.save(output_path)
        print(f"  Saved adapted model → {output_path}")


# ── Run Phase 1 ───────────────────────────────────────────────────────────────
if CFG["load_da"]:
    print(f"[Phase 1] Loading saved checkpoint: {CFG['load_da']}")
    sbert = SentenceTransformer(CFG["load_da"])
elif not CFG["skip_da"]:
    print("[Phase 1] MLM Domain Adaptation")
    all_texts = train_df["TEXT"].astype(str).tolist() + test_df["TEXT"].astype(str).tolist()
    domain_adapt_mlm(
        sbert, all_texts,
        epochs=CFG["da_epochs"], batch_size=CFG["da_batch"],
        lr=CFG["da_lr"], mlm_prob=CFG["da_mlm_prob"],
        output_path=CFG["da_save"],
    )
else:
    print("[Phase 1] Skipped — using base model weights")

# Save Phase-1 base for CV reuse
sbert_phase1 = sbert

if CFG["skip_ft"]:
    print("[skip_ft=True] Exiting after Phase 1.")


[Phase 1] MLM Domain Adaptation
  Texts    : 13000 (Train + Test, no labels)
  Epochs   : 3  |  batch=8  |  lr=3e-05
  MLM prob : 0.15


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
MPNetForMaskedLM LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                       | Status     | 
--------------------------+------------+-
embeddings.position_ids   | UNEXPECTED | 
pooler.dense.bias         | UNEXPECTED | 
pooler.dense.weight       | UNEXPECTED | 
lm_head.decoder.bias      | MISSING    | 
lm_head.bias              | MISSING    | 
lm_head.layer_norm.bias   | MISSING    | 
lm_head.dense.bias        | MISSING    | 
lm_head.layer_norm.weight | MISSING    | 
lm_head.dense.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your

    epoch 1/3  step 50/1625  loss=11.0517
    epoch 1/3  step 100/1625  loss=10.0457
    epoch 1/3  step 150/1625  loss=9.4876
    epoch 1/3  step 200/1625  loss=9.0104
    epoch 1/3  step 250/1625  loss=8.6188
    epoch 1/3  step 300/1625  loss=8.3312
    epoch 1/3  step 350/1625  loss=8.0747
    epoch 1/3  step 400/1625  loss=7.8573
    epoch 1/3  step 450/1625  loss=7.6467
    epoch 1/3  step 500/1625  loss=7.4460
    epoch 1/3  step 550/1625  loss=7.2751
    epoch 1/3  step 600/1625  loss=7.0988
    epoch 1/3  step 650/1625  loss=6.9593
    epoch 1/3  step 700/1625  loss=6.8137
    epoch 1/3  step 750/1625  loss=6.6900
    epoch 1/3  step 800/1625  loss=6.5865
    epoch 1/3  step 850/1625  loss=6.4808
    epoch 1/3  step 900/1625  loss=6.3848
    epoch 1/3  step 950/1625  loss=6.3019
    epoch 1/3  step 1000/1625  loss=6.2083
    epoch 1/3  step 1050/1625  loss=6.1271
    epoch 1/3  step 1100/1625  loss=6.0627
    epoch 1/3  step 1150/1625  loss=5.9918
    epoch 1/3  step 1200/1625

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved adapted model → /content/drive/MyDrive/Colab_Notebooks/a5/v8_da_model.pt


## Step 3 — Phase 2 Helpers: Supervised Fine-tuning & Embedding

以下 helper 函式在 CV 每個 round 內部被呼叫，定義於此以便閱讀時對照。

**CFG keys used:** `ft_pairs`, `ft_epochs`, `ft_batch`, `ft_lr`, `ft_warmup`, `ft_save`

In [19]:
# ── Helper: sample sentence pairs for CosineSimilarityLoss ──────────────
def _sample_pairs(df: pd.DataFrame, n_pairs: int, seed: int = 42) -> list:
    rng    = random.Random(seed)
    texts  = df["TEXT"].astype(str).tolist()
    labels = df["LABEL"].astype(int).tolist()

    pos_idx = [i for i, l in enumerate(labels) if l == 1]
    neg_idx = [i for i, l in enumerate(labels) if l == 0]

    examples, half = [], n_pairs // 2
    for _ in range(half):
        pool = pos_idx if rng.random() < 0.5 else neg_idx
        a, b = rng.sample(pool, 2) if len(pool) >= 2 else (pool[0], pool[0])
        examples.append(InputExample(texts=[texts[a], texts[b]], label=1.0))
    for _ in range(n_pairs - half):
        a, b = rng.choice(pos_idx), rng.choice(neg_idx)
        examples.append(InputExample(texts=[texts[a], texts[b]], label=0.0))
    rng.shuffle(examples)
    return examples


# ── Helper: fine-tune SBERT with CosineSimilarityLoss ────────────────────
def finetune_sbert_supervised(
    sbert: SentenceTransformer,
    train_df: pd.DataFrame,
    n_pairs: int      = 3000,   # CFG["ft_pairs"]
    epochs: int       = 3,      # CFG["ft_epochs"]
    batch_size: int   = 16,     # CFG["ft_batch"]
    lr: float         = 2e-5,   # CFG["ft_lr"]
    warmup_steps: int = 100,    # CFG["ft_warmup"]
    output_path       = None,   # CFG["ft_save"] (only for final retrain)
) -> None:
    print(f"  Fine-tuning on {len(train_df)} rows  "
          f"pairs={n_pairs}  epochs={epochs}  lr={lr}")

    examples   = _sample_pairs(train_df, n_pairs)
    dataloader = DataLoader(examples, shuffle=True, batch_size=batch_size,
                            collate_fn=lambda b: b)
    loss_fn    = losses.CosineSimilarityLoss(sbert)

    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sbert.to(device).train()

    optimizer   = torch.optim.AdamW(sbert.parameters(), lr=lr)
    total_steps = len(dataloader) * epochs
    scheduler   = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=min(warmup_steps, total_steps)
    )

    global_step = 0
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for step, batch in enumerate(dataloader, 1):
            texts_a = [ex.texts[0] for ex in batch]
            texts_b = [ex.texts[1] for ex in batch]
            labels  = torch.tensor([ex.label for ex in batch],
                                   dtype=torch.float, device=device)
            feats_a = {k: v.to(device) for k, v in sbert.tokenize(texts_a).items()
                       if isinstance(v, torch.Tensor)}
            feats_b = {k: v.to(device) for k, v in sbert.tokenize(texts_b).items()
                       if isinstance(v, torch.Tensor)}
            loss = loss_fn([feats_a, feats_b], labels)
            loss.backward()
            optimizer.step()
            if global_step < warmup_steps:
                scheduler.step()
            optimizer.zero_grad()
            total_loss  += loss.item()
            global_step += 1
            if step % 50 == 0:
                print(f"    epoch {epoch}/{epochs}  step {step}/{len(dataloader)}"
                      f"  loss={total_loss/step:.4f}")
        print(f"  Epoch {epoch} done — avg loss: {total_loss/len(dataloader):.4f}")

    sbert.train(False)
    if output_path:
        sbert.save(output_path)
        print(f"  Saved fine-tuned model → {output_path}")


# ── Helper: extract SBERT embeddings row-by-row ───────────────────────────
def run_row_by_row(
    df: pd.DataFrame,
    sbert: SentenceTransformer,
    has_label: bool = True,
) -> pd.DataFrame:
    total   = len(df)
    records = []
    print(f"  Extracting embeddings for {total} rows …")
    for i, (_, row) in enumerate(df.iterrows()):
        if i % 200 == 0:
            print(f"    progress: {i}/{total}")
        emb    = sbert.encode(str(row["TEXT"]), normalize_embeddings=True,
                              show_progress_bar=False).astype("float32")
        label  = int(row["LABEL"]) if has_label else -1
        record = {"row_id": int(row["row_id"]), "LABEL": label}
        record.update({f"emb_{j}": round(float(v), 6) for j, v in enumerate(emb)})
        records.append(record)
    result_df = pd.DataFrame(records)[["row_id", "LABEL"] + AGG_FEATURES]
    print(f"  Done — {len(result_df)} rows  (dim={len(ALL_FEATURES)})")
    return result_df


## Step 4 — CV: 5 × 80/20 with Fine-tune Inside Each Round

每個 round 從 Phase-1 base model 重新複製，僅用 80% train 做 Phase-2 fine-tune，
再對 train/val 分別取 embedding，最後訓練 ML 模型並在 val 上評估。

**CFG keys used:** `model` (`rf`/`elastic`/`gbm`)

In [ ]:
# ── Helper: build sklearn ML model ──────────────────────────────────────
def _make_model(model_name: str):
    model_map = {
        "rf":      RandomForestRegressor(n_estimators=100, random_state=42),
        "elastic": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
        "gbm":     GradientBoostingRegressor(n_estimators=100, random_state=42),
    }
    if model_name not in model_map:
        raise ValueError(f"Unknown model '{model_name}'. Choose from: {list(model_map)}")
    return model_map[model_name]


# ── Helper: print per-round metrics ──────────────────────────────────────
def _print_fold_metrics(fold: int, y_true, y_pred_label, y_score) -> dict:
    cm  = confusion_matrix(y_true, y_pred_label)
    acc = accuracy_score(y_true, y_pred_label)
    pre = precision_score(y_true, y_pred_label, zero_division=0)
    rec = recall_score(y_true, y_pred_label, zero_division=0)
    f1  = f1_score(y_true, y_pred_label, zero_division=0)
    mse = mean_squared_error(y_true, y_score)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = float("nan")
    print(f"\n  ── Round {fold} ──")
    print(f"  Confusion Matrix:\n{cm}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {pre:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print(f"  AUC      : {auc:.4f}  MSE: {mse:.4f}")
    return dict(acc=acc, pre=pre, rec=rec, f1=f1, auc=auc, mse=mse, cm=cm)


# ── 5 × 80/20 CV (Phase-2 fine-tune inside each round) ───────────────────
y       = train_df["LABEL"].astype(int).values
row_ids = train_df["row_id"].astype(int).values

sss          = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
fold_metrics = []
oof_scores   = np.full(len(y), np.nan)

for fold, (train_idx, val_idx) in enumerate(sss.split(np.zeros(len(y)), y), start=1):
    print(f"\n══ Round {fold}/5  (train={len(train_idx)}, val={len(val_idx)}) ══")

    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df   = train_df.iloc[val_idx].reset_index(drop=True)

    # deepcopy Phase-1 base → each round starts from the same encoder
    sbert_fold = copy.deepcopy(sbert_phase1)

    # Phase 2: fine-tune on 80% train split only (no val leakage)
    finetune_sbert_supervised(
        sbert_fold, fold_train_df,
        n_pairs=CFG["ft_pairs"], epochs=CFG["ft_epochs"],
        batch_size=CFG["ft_batch"], lr=CFG["ft_lr"],
        warmup_steps=CFG["ft_warmup"], output_path=None,
    )

    # Embed train and val separately with the fold-specific model
    train_emb_df = run_row_by_row(fold_train_df, sbert_fold, has_label=True)
    val_emb_df   = run_row_by_row(fold_val_df,   sbert_fold, has_label=True)

    X_tr  = train_emb_df[ALL_FEATURES].values
    y_tr  = train_emb_df["LABEL"].values
    X_val = val_emb_df[ALL_FEATURES].values
    y_val = val_emb_df["LABEL"].values

    pipe = Pipeline([("scaler", StandardScaler()), ("model", _make_model(CFG["model"]))])
    pipe.fit(X_tr, y_tr)
    scores              = np.clip(pipe.predict(X_val), 0, 1)
    oof_scores[val_idx] = scores
    fold_metrics.append(_print_fold_metrics(fold, y_val, (scores >= 0.5).astype(int), scores))

oof_labels = np.where(~np.isnan(oof_scores), (oof_scores >= 0.5).astype(int), -1)

print("\n══ Overall (mean ± std across 5 rounds) ══")
for metric in ("acc", "pre", "rec", "f1", "auc", "mse"):
    vals = [m[metric] for m in fold_metrics]
    print(f"  {metric.upper():9s}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}")



══ Round 1/5  (train=1600, val=400) ══
  Fine-tuning on 1600 rows  pairs=3000  epochs=3  lr=2e-05
    epoch 1/3  step 50/188  loss=0.2598
    epoch 1/3  step 100/188  loss=0.2552
    epoch 1/3  step 150/188  loss=0.2438
  Epoch 1 done — avg loss: 0.2319
    epoch 2/3  step 50/188  loss=0.1331
    epoch 2/3  step 100/188  loss=0.1189
    epoch 2/3  step 150/188  loss=0.1140
  Epoch 2 done — avg loss: 0.1090
    epoch 3/3  step 50/188  loss=0.0514
    epoch 3/3  step 100/188  loss=0.0487
    epoch 3/3  step 150/188  loss=0.0471
  Epoch 3 done — avg loss: 0.0480
  Extracting embeddings for 1600 rows …
    progress: 0/1600
    progress: 200/1600
    progress: 400/1600
    progress: 600/1600
    progress: 800/1600
    progress: 1000/1600
    progress: 1200/1600
    progress: 1400/1600
  Done — 1600 rows  (dim=768)
  Extracting embeddings for 400 rows …
    progress: 0/400
    progress: 200/400
  Done — 400 rows  (dim=768)

  ── Round 1 ──
  Confusion Matrix:
[[169  31]
 [ 38 162]]
  Accura

: 

## Step 5 — Final Model: Retrain on Full Training Data

CV 結束後，用全部 training data 再做一次 Phase-2 fine-tune，訓練最終 ML 模型，
供 test set 推論使用。

In [ ]:
print("Retraining Phase-2 on full training data …")
sbert_final = copy.deepcopy(sbert_phase1)
finetune_sbert_supervised(
    sbert_final, train_df,
    n_pairs=CFG["ft_pairs"], epochs=CFG["ft_epochs"],
    batch_size=CFG["ft_batch"], lr=CFG["ft_lr"],
    warmup_steps=CFG["ft_warmup"], output_path=CFG["ft_save"],
)

full_emb_df  = run_row_by_row(train_df, sbert_final, has_label=True)
X_full       = full_emb_df[ALL_FEATURES].values
final_pipe   = Pipeline([("scaler", StandardScaler()), ("model", _make_model(CFG["model"]))])
final_pipe.fit(X_full, y)
print("Final model trained.")


## Step 6 — Test Set Inference & Save

In [ ]:
test_emb_df = run_row_by_row(test_df, sbert_final, has_label=False)

X_test      = test_emb_df[ALL_FEATURES].values
test_scores = np.clip(final_pipe.predict(X_test), 0, 1)
test_labels = (test_scores >= 0.5).astype(int)

test_output = test_emb_df[["row_id"]].copy()
test_output["LABEL"] = test_labels
test_output.to_csv(CFG["test_output"], index=False)
print(f"Saved {len(test_output)} predictions → {CFG['test_output']}")
